In [4]:
!pip install datasets

In [5]:
from datasets import load_dataset

ds = load_dataset("knkarthick/dialogsum")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [7]:
ds['train'][1]

{'id': 'train_1',
 'dialogue': "#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little.",
 'summary': 'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.',
 'topic': 'vaccines'}

In [8]:
ds['train'][4]

{'id': 'train_4',
 'dialogue': "#Person1#: Watsup, ladies! Y'll looking'fine tonight. May I have this dance?\n#Person2#: He's cute! He looks like Tiger Woods! But, I can't dance. . .\n#Person1#: It's all good. I'll show you all the right moves. My name's Malik.\n#Person2#: Nice to meet you. I'm Wen, and this is Nikki.\n#Person1#: How you feeling', vista? Mind if I take your friend'round the dance floor?\n#Person2#: She doesn't mind if you don't mind getting your feet stepped on.\n#Person1#: Right. Cool! Let's go!",
 'summary': "Malik invites Nikki to dance. Nikki agrees if Malik doesn't mind getting his feet stepped on.",
 'topic': 'dance'}

In [9]:
ds['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [10]:
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

#**1.Without Fine Tune**

In [11]:
pip install transformers==4.40.0

In [12]:
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
article = ds['train'][1]['dialogue']

In [14]:
article

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [15]:
pipe(article,max_length=25,min_length=15,do_sample=False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old'}]

In [16]:
pipe(article,max_length=45,min_length=15,do_sample=False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for hepatitis A, Chickenpox and Measles shots.'}]

#**2.With Fine Tuning**

In [17]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [18]:
# Tokenization
def preprocess_function(batch):
  source = batch['dialogue']
  target = batch['summary']
  source_id = tokenizer(source,padding="max_length",truncation=True,max_length=128)
  target_id = tokenizer(target,padding="max_length",truncation=True,max_length=128)

  labels=target_id['input_ids']
  labels= [[(label if label != tokenizer.pad_token_id else -100) for label in labels_example] for labels_example in labels]

  return{
      "input_ids":source_id["input_ids"],
      "attention_mask":source_id["attention_mask"],
      "labels":labels
  }

In [19]:
df_source = ds.map(preprocess_function,batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [20]:
pip uninstall transformers peft accelerate -y

Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: accelerate 0.30.0
Uninstalling accelerate-0.30.0:
  Successfully uninstalled accelerate-0.30.0


In [21]:
pip install transformers==4.40.0 peft==0.10.0 accelerate==0.30.0

  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached accelerate-0.30.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.40.0-py3-none-any.whl (9.0 MB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)
Using cached accelerate-0.30.0-py3-none-any.whl (302 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [22]:
from transformers import TrainingArguments, Trainer

training_arguments = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    remove_unused_columns = True,
    report_to="none"   # ✅ disables wandb(weight and bias)
)

In [23]:
trainer = Trainer(
    model = model,
    args = training_arguments,
    train_dataset = df_source['train'],
    eval_dataset = df_source['test']
)

In [24]:
trainer.train()

Step,Training Loss
500,1.591000
1000,1.486900
1500,1.434200
2000,1.082400
2500,1.016800
3000,0.999200


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_

TrainOutput(global_step=3116, training_loss=1.25956239810498, metrics={'train_runtime': 3383.9438, 'train_samples_per_second': 7.364, 'train_steps_per_second': 0.921, 'total_flos': 6750530835578880.0, 'train_loss': 1.25956239810498, 'epoch': 2.0})

In [25]:
eval_results = trainer.evaluate()

In [26]:
eval_results

{'eval_loss': 1.6870027780532837,
 'eval_runtime': 49.8257,
 'eval_samples_per_second': 30.105,
 'eval_steps_per_second': 3.773,
 'epoch': 2.0}

In [27]:
model.save_pretrained('/content/model_directory')
tokenizer.save_pretrained('/content/model_directory')

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


('/content/model_directory/tokenizer_config.json',
 '/content/model_directory/special_tokens_map.json',
 '/content/model_directory/vocab.json',
 '/content/model_directory/merges.txt',
 '/content/model_directory/added_tokens.json',
 '/content/model_directory/tokenizer.json')

In [28]:
tokenizer = AutoTokenizer.from_pretrained('/content/model_directory')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/model_directory')

def summarize(blog_post):
  # tokenize the input blog post
  inputs = tokenizer(blog_post, max_length=1024, truncation=True, return_tensors='pt')

  # generate the summary
  summary_ids = model.generate(inputs["input_ids"],max_length=150,min_length=40,length_penalty=2.0,num_beams=4,early_stopping=True)

  # decode the summary
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

  # clean output
  summary = summary.replace("#", "").strip()

  return summary

In [29]:
blog_post="""
Machine learning is a transformative branch of artificial intelligence that enables computers to learn from data and improve
their performance without being explicitly programmed. It plays a crucial role in modern technology, powering applications
across industries such as healthcare, finance, education, and entertainment. By analyzing vast amounts of structured and
unstructured data, machine learning algorithms can detect patterns, make predictions, and automate complex decision-making
processes. For example, in healthcare, machine learning models assist doctors in diagnosing diseases at early stages,
while in finance, they are used for fraud detection and risk assessment. Recommendation systems used by streaming platforms
and e-commerce websites also rely heavily on machine learning to provide personalized user experiences.

There are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning.
Supervised learning involves training a model on labeled data, where the correct output is known, while unsupervised learning
deals with unlabeled data and aims to find hidden patterns or groupings. Reinforcement learning, on the other hand, focuses on
training agents to make decisions by rewarding desired behaviors. With the advancement of deep learning, a subset of machine
learning that uses neural networks with multiple layers, the field has seen significant breakthroughs in areas like computer
vision, speech recognition, and natural language processing.

Despite its many advantages, machine learning also presents challenges. Issues such as data privacy, algorithmic bias, and
lack of transparency can impact the reliability and fairness of models. Additionally, training complex models often requires
large computational resources and high-quality data. As a result, researchers and practitioners are actively working on
developing more efficient, interpretable, and ethical machine learning systems. Overall, machine learning continues to evolve
rapidly, shaping the future of technology and offering innovative solutions to real-world problems.
"""
summary=summarize(blog_post)

print(f"Summary : {summary}")

Summary : Machinelearning is a transformative branch of artificial intelligence that enables computers to learn from data and improve performance without being explicitly programmed. It plays a crucial role in modern technology and there are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning. Machinelearning challenges include data privacy, algorithmic bias, and lack of transparency. Reinforcement learning is a subset of machinelearning that uses neural networks with multiple layers, while Supervised learning is based on deep learning.


In [30]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model
model_path = "/content/model_directory"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Summarization function
def summarize(text):
    if not text.strip():
        return "Please enter some text."

    text = "summarize: " + text

    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # clean output
    summary = summary.replace("#", "").strip()

    return summary


# UI Layout
with gr.Blocks() as app:

    gr.Markdown("# 📝 AI Text Summarization App")
    gr.Markdown("Enter a long paragraph and get a short summary")

    # Input box (full width)
    input_text = gr.Textbox(
        lines=12,
        placeholder="Enter your text here...",
        label="Input Text"
    )

    # Buttons row
    with gr.Row():
        submit_btn = gr.Button("🚀 Summarize")
        clear_btn = gr.Button("🧹 Clear")

    # Output box (full width)
    output_text = gr.Textbox(
        lines=10,
        label="Summary"
    )

    # Button actions
    submit_btn.click(fn=summarize, inputs=input_text, outputs=output_text)

    clear_btn.click(lambda: ("", ""), None, [input_text, output_text])


# Launch app
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd8eeac2b6897bc8f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
import shutil
shutil.make_archive("Model","zip","/content/model_directory")

'/content/Model.zip'

In [33]:
from google.colab import files
files.download("Model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>